# Kaggle training run: English → Hindi

Run these cells in order in a Kaggle Python notebook. In Notebook Settings, enable **Internet** and choose the GPU accelerator. This workflow uses OPUS-100, trains the basic Transformer and mT5, prompts LLaMA and Gemma, evaluates both at multiple temperatures, and packages only report files and predictions (not datasets or model weights).

## Before running

1. Open Notebook Settings → Session options. Enable Internet and select GPU; restart the session if Kaggle prompts you.
2. For LLaMA/Gemma, accept their access terms on Hugging Face first. Add a Kaggle Secret named `HF_TOKEN` with a read token (Add-ons → Secrets) and enable it for this notebook. Never paste the token into a cell.
3. The training stages may take hours. If time is short, run the baseline and mT5 first, save the notebook version, and run the decoder-only cells in a second session.

In [ ]:
from pathlib import Path
import os, subprocess, torch

repo_dir = Path('/kaggle/working/generative-ai-machine-translation')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/saishthakur/generative-ai-machine-translation.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository:', Path.cwd())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Turn on a Kaggle GPU accelerator, restart the session, and rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Keep Kaggle's preinstalled CUDA-enabled PyTorch; install the other project dependencies.
%pip install -q 'transformers>=4.46,<6' 'datasets>=3.0' 'sentencepiece>=0.2' 'sacrebleu>=2.4' 'accelerate>=0.34'


In [ ]:
import torch, transformers, datasets, sacrebleu
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__, '| sacrebleu:', sacrebleu.__version__)
!nvidia-smi


In [ ]:
# Optional HF authentication for gated LLaMA and Gemma model checkpoints.
# Add a Kaggle Secret named HF_TOKEN first. Do not print or paste its value.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    del hf_token
    print('Hugging Face authentication is ready.')
except Exception:
    print('No HF_TOKEN secret found. OPUS-100 and public checkpoints may work; gated LLaMA/Gemma access may fail.')


## 1. Download and prepare the same fixed dataset splits

The script uses the corpus-provided validation and test splits, samples 20,000 training pairs with seed 4403, and writes local CSVs under `data/processed/`. Do not upload those files unless the dataset terms permit it.

In [ ]:
!python scripts/prepare_data.py --train-rows 20000 --validation-rows 1000 --test-rows 1000 --seed 4403
!python -c "import csv, pathlib; p=pathlib.Path('data/processed'); [print(n, sum(1 for _ in csv.DictReader((p/f'{n}.csv').open(encoding='utf-8')))) for n in ['train','validation','test']]"


## 2. Train the basic encoder-decoder Transformer from scratch

This initializes all model weights randomly; only the ByT5 tokenizer is reused. If this runs out of time or memory, reduce the batch size to 4.

In [ ]:
!python scripts/train_basic_transformer.py --epochs 5 --batch-size 8 --max-length 256


## 3. Fine-tune mT5-small

This is full fine-tuning. The batch size and gradient checkpointing are conservative for a single GPU.

In [ ]:
!python scripts/train_pretrained.py --architecture seq2seq --model google/mt5-small --output-dir outputs/mt5-small --epochs 1 --batch-size 2 --max-length 128 --gradient-checkpointing


## 4. Evaluate the two encoder-decoder systems

Both use the same 1,000-row test CSV. Evaluation writes a prediction CSV and a metric JSON for each model.

In [ ]:
!python scripts/evaluate.py --architecture seq2seq --model outputs/basic-transformer --batch-size 8 --max-input-length 256 --max-new-tokens 256 --output-dir outputs/evaluation/basic-transformer
!python scripts/evaluate.py --architecture seq2seq --model outputs/mt5-small --batch-size 4 --max-input-length 128 --max-new-tokens 128 --output-dir outputs/evaluation/mt5-small


## 5. Prompt LLaMA and Gemma

These use instruction-tuned small checkpoints and the prompt templates in `scripts/evaluate.py`. If access is denied, confirm both the model terms and the `HF_TOKEN` secret are enabled for this notebook.

In [ ]:
!python scripts/evaluate.py --architecture causal --model meta-llama/Llama-3.2-1B-Instruct --batch-size 2 --max-input-length 256 --max-new-tokens 128 --temperature 0 --output-dir outputs/evaluation/llama-t0
!python scripts/evaluate.py --architecture causal --model google/gemma-3-1b-it --batch-size 2 --max-input-length 256 --max-new-tokens 128 --temperature 0 --output-dir outputs/evaluation/gemma-t0


## 6. Sampling-temperature analysis

Run the decoder-only systems at 0.3 and 0.7. Compare the resulting BLEU/chrF scores and inspect prediction CSVs for fluency, consistency, and faithfulness.

In [ ]:
!python scripts/evaluate.py --architecture causal --model meta-llama/Llama-3.2-1B-Instruct --batch-size 2 --max-input-length 256 --max-new-tokens 128 --temperature 0.3 --output-dir outputs/evaluation/llama-t03
!python scripts/evaluate.py --architecture causal --model meta-llama/Llama-3.2-1B-Instruct --batch-size 2 --max-input-length 256 --max-new-tokens 128 --temperature 0.7 --output-dir outputs/evaluation/llama-t07
!python scripts/evaluate.py --architecture causal --model google/gemma-3-1b-it --batch-size 2 --max-input-length 256 --max-new-tokens 128 --temperature 0.3 --output-dir outputs/evaluation/gemma-t03
!python scripts/evaluate.py --architecture causal --model google/gemma-3-1b-it --batch-size 2 --max-input-length 256 --max-new-tokens 128 --temperature 0.7 --output-dir outputs/evaluation/gemma-t07


In [ ]:
!python scripts/summarize_results.py --outputs-dir outputs --output outputs/results_summary.csv
from IPython.display import display
import pandas as pd
display(pd.read_csv('outputs/results_summary.csv'))


## 7. Package results to bring back to GitHub

This copies only the summary, metric JSONs, training metric JSONs, and generated predictions. It excludes the downloaded dataset and model checkpoints. Download `assignment_results.zip` from the Kaggle notebook Output panel and attach it in the Codex chat; Codex can then add the measured results to the repository.

In [ ]:
from pathlib import Path
import shutil

out = Path('outputs')
bundle = Path('/kaggle/working/results_for_github')
if bundle.exists():
    shutil.rmtree(bundle)
for p in out.rglob('*_metrics.json'):
    dest = bundle / p.relative_to(out)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dest)
for p in out.rglob('training_metrics.json'):
    dest = bundle / p.relative_to(out)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dest)
for p in (out / 'evaluation').rglob('*_predictions.csv'):
    dest = bundle / p.relative_to(out)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dest)
shutil.copy2(out / 'results_summary.csv', bundle / 'results_summary.csv')
archive = shutil.make_archive('/kaggle/working/assignment_results', 'zip', bundle)
print('Download this file from Kaggle Outputs:', archive)
from IPython.display import FileLink, display
display(FileLink(archive))
